# Week 4: LLM Benchmarking vs Week 3 ML

Cells in this notebook were taken from the tutorial written by Victoria Reynolds.

Steps:
- Load Week 3 test data
- Query LLM endpoint
- Parse outputs into structured predictions
- Compare ML vs each LLM
- Save raw, clean, and summary result CSV files

## Step 1: Imports

In [5]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from datetime import datetime

openai = __import__('openai')
OpenAI = openai.OpenAI

## Step 2: Paths and settings

In [43]:
ROOT = Path('..').resolve()

WEEK3_TEST_CSV = 'week3_artifacts/FTES_1hour_test.csv'
WEEK3_ML_RESULTS_CSV = 'week3_artifacts/xgb_test_predictions.csv'
WEEK3_ML_CONFIG_JSON = 'week3_artifacts/xgb_training_config.json'

with open(WEEK3_ML_CONFIG_JSON, "r", encoding="utf-8") as f:
    train_cfg = json.load(f)

TIME_COL = 'Time'
FEAT_COLS = train_cfg.get('feature_cols')
TRUE_VAL_COL = 'Injection Pressure'
ML_PRED_COL = 'Injection Pressure__pred'

# Only using gemma-3-12b-it
LLM_MODEL = "gemma3"
MODEL_ENDPOINTS = [  
    # {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    # {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

PROMPT_DIR = ROOT / 'Prompts'
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_PATH = OUTPUT_DIR / f'{LLM_MODEL}_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = OUTPUT_DIR / f'{LLM_MODEL}_results_clean_{ITERATION_NUMBER}.csv'
SUMMARY_RESULTS_PATH = OUTPUT_DIR / f'{LLM_MODEL}_results_summary_{ITERATION_NUMBER}.csv'

## Step 3: Load Week 3 files

In [7]:
test_df = pd.read_csv(WEEK3_TEST_CSV)
ml_df = pd.read_csv(WEEK3_ML_RESULTS_CSV)

needed_test = {TIME_COL, *FEAT_COLS, TRUE_VAL_COL}
needed_ml = {TIME_COL, ML_PRED_COL}
if needed_test - set(test_df.columns):
    raise ValueError(f'Missing in {WEEK3_TEST_CSV}: {sorted(needed_test - set(test_df.columns))}')
if needed_ml - set(ml_df.columns):
    raise ValueError(f'Missing in {WEEK3_ML_RESULTS_CSV}: {sorted(needed_ml - set(ml_df.columns))}')

df = test_df[[TIME_COL, *FEAT_COLS, TRUE_VAL_COL]].merge(
    ml_df[[TIME_COL, ML_PRED_COL]], on=TIME_COL, how='left'
)

print(f'Loaded rows in test set: {len(df)}')
# display(df.head())

Loaded rows in test set: 256


## Step 4: Prompt + parser helpers

What this does: defines a strict prompt format and a parser that turns model text into structured fields (prediction, confidence, parse status).

Why it matters: LLM responses are free-form by default, but evaluation requires deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear output contracts plus defensive checks.

In [8]:
SYSTEM_PROMPT = """You are a careful Fracture Thermal Energy Storage time-series forecasting model. \n
Your task is to predict the Target metric at t+1 using only features measured at time t. \n
Do not use future information, external knowledge, or assumptions not present in the input. \n
Treat this as numeric regression, not classification. \n
Return exactly one JSON object and nothing else. \n\n

Rules: \n
Prediction must be in the correct units for the target. \n
Confidence must be in [0, 1]. \n
No markdown, no prose, no code fences, no extra keys. \n
Required JSON schema: \n
{
"t+1": <M/d/yyyy h:mm:ss a>
"prediction": <float>, 
"confidence": <0_to_1_float>
}
"""

few_shot_samples = [
    {
        't': "2/11/2025  11:00:00 PM",
        'input_features': {
            "Injection Pressure": 4981.6798,
            "Pressure_Differential": 4353.3743,
            "TC Interval Pressure": 4991.9235,
            "TN Interval Flow": 0.0008,
            "TL Collar Flow": 0.1707,
            "TC-TEC-INT-L": 43.2451,
            "TC-TEC-INT-L__lag_12h": 43.3222,
            "TC-TEC-INT-L__lag_24h": 43.1582,
            "TC-TEC-INT-L__roll_mean_12h": 43.3166,
            "TC-TEC-INT-L__roll_mean_24h": 43.2913, 
            "TN-TEC-BOT-U": 30.4198,
            "TN-TEC-BOT-U__lag_12h": 30.4127,
            "TN-TEC-BOT-U__lag_24h": 30.3488,
            "TN-TEC-BOT-U__roll_mean_12h": 30.4158,
            "TN-TEC-BOT-U__roll_mean_24h": 30.3955
        },
        'output_json': {'t+1': "2/12/2025  12:00:00 AM", 'prediction': 4982.9503, 'confidence': 0.82}
    },
    {
        't': "2/12/2025  12:00:00 AM",
        'input_features': {
            "Injection Pressure": 4982.9503,
            "Pressure_Differential": 4354.5854,
            "TC Interval Pressure": 4992.9041,
            "TN Interval Flow": 0.0009,
            "TL Collar Flow": 0.1708,
            "TC-TEC-INT-L": 43.2164,
            "TC-TEC-INT-L__lag_12h": 43.3474,
            "TC-TEC-INT-L__lag_24h": 43.1934,
            "TC-TEC-INT-L__roll_mean_12h": 43.3102,
            "TC-TEC-INT-L__roll_mean_24h": 43.2949, 
            "TN-TEC-BOT-U": 30.4324,
            "TN-TEC-BOT-U__lag_12h": 30.4153,
            "TN-TEC-BOT-U__lag_24h": 30.1394,
            "TN-TEC-BOT-U__roll_mean_12h": 30.4164,
            "TN-TEC-BOT-U__roll_mean_24h": 30.3985
        },
        'output_json': {'t+1': "2/12/2025  1:00:00 AM", 'prediction': 4982.1107, 'confidence': 0.91}
    }
]

def build_prompt(time_t, feature_names, features_t, version='v1', few_shot=None):
    features_json = json.dumps(feature_names)
    features_t = json.dumps(features_t)

    if version == 'v1':
        prompt_lines = [
            'Forecast target: Injection Pressure at next 1 hour interval (t+1h)',
            f'Current timestamp: {time_t}',
            'Horizon for t+1: 1 hour from current timestamp', 
            'Target for prediction: Injection Pressure',
            'Units for prediction: pressure in psi',
            f'Use only the following feature snapshot at time t: {features_json}',
            'Important: Do not use future information to make predictions. Return JSON only. No explanation text.',
            'Required output schema: {\"t+1\": <M/d/yyyy h:mm:ss a>, \"prediction\": <float>, \"confidence\": <0_to_1_float>}'
        ]
            
    if few_shot:
        for i, ex in enumerate(few_shot, start=1):
            prompt_lines.append('')
            prompt_lines.append(f'Example input at time t, {ex["t"]}: {ex["input_features"]}')
            prompt_lines.append(f'Example output for prediction at time t+1h: {json.dumps(ex["output_json"])}')
    prompt_lines.extend([
        '',
        'Input:',
        features_t
    ])
    return '\n'.join(prompt_lines)

    raise ValueError(f'Unknown version: {version}')
    

### Parse response

In [9]:
def parse_response(raw_text):
    output = {'t+1': None, 'llm_prediction': np.nan, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        output['parse_error'] = 'invalid_json'
        return output

    t_plus_1 = payload.get('t+1', None)
#     try:
#         t_plus_1 = datetime.strptime(t_plus_1, "%m/%d/%Y %I:%M:%S %p")
#     except Exception:
#         t_plus_1 = None

    pred = payload.get('prediction', np.nan)
    try:
        pred = float(pred)
    except Exception:
        pred = np.nan

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    if not np.isnan(conf) and not (0 <= conf <= 1):
        conf = np.nan
    
    output['t+1'] = t_plus_1
    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output

## Step 5: Single-row smoke test

What this does: tests one example on each model endpoint before running the full benchmark loop.

Why it matters: this catches endpoint/port issues early and confirms all models are reachable.

Principle: validate connectivity and output format for each model before large-scale evaluation.

In [ ]:
# Configure prompt version to run/save
selected_prompt_ver = 'v1'

In [10]:
def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

example = df.iloc[0]
for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(build_prompt(example['Time'], 
                                 FEAT_COLS, 
                                 example[FEAT_COLS].to_dict(), 
                                 version=selected_prompt_ver,
                                 few_shot=few_shot_samples), 
                    endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw)
    print(f"\nModel: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print('Raw response:')
    print(raw)
    print('Parsed:')
    print(parsed)


Model: gemma-3-12b-it (port 8002)
Raw response:
```json
{
"t+1": "2/12/2025 12:00:00 AM",
"prediction": 4982.4312,
"confidence": 0.88
}
```
Parsed:
{'t+1': '2/12/2025 12:00:00 AM', 'llm_prediction': 4982.4312, 'llm_confidence': 0.88, 'parse_ok': True, 'parse_error': None}


### Save prompts

In [11]:
prompt_example = build_prompt("{time_t}", FEAT_COLS, {feat: "{float}" for feat in FEAT_COLS}, version=selected_prompt_ver, few_shot=few_shot_samples)
prompt_template_path = PROMPT_DIR / f'prompt_template_{selected_prompt_ver}.txt'
prompt_template_path.write_text(prompt_example, encoding='utf-8')


few_shot_path = PROMPT_DIR / f'few_shot_samples_{selected_prompt_ver}.json'
few_shot_path.write_text(json.dumps(few_shot_samples, indent=2), encoding='utf-8')

1581

## Step 6: Full benchmark run

What this does: loops over all configured model endpoints and all test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset and prompt constant while varying models for a fair model-to-model comparison.

In [35]:
rows = []
for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning benchmark for {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
    for i, row in df.iterrows():
        raw = query_llm(
            build_prompt(example['Time'], FEAT_COLS, example[FEAT_COLS].to_dict(), version=selected_prompt_ver, few_shot=few_shot_samples),
            endpoint_cfg,
            temperature=0.0,
            seed=0
        )
        parsed = parse_response(raw)
        
        if isinstance(row[TRUE_VAL_COL], float):
            ground_truth = row[TRUE_VAL_COL]
        else:
            ground_truth = row[TRUE_VAL_COL].iloc[0]

        rows.append({
            TIME_COL: row[TIME_COL],
            'model_label': endpoint_cfg['label'],
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'prompt_version': selected_prompt_ver,
            'raw_response': raw,
            **parsed,
            'ground_truth': ground_truth,
            'ml_prediction': row[ML_PRED_COL]
        })

        if (i + 1) % 10 == 0:
            print(f"  Completed {i + 1}/{len(df)}")
            
    print(f"{endpoint_cfg['model_name']} benchmark run complete.")


Running benchmark for gemma-3-12b-it on port 8002...
  Completed 10/256
  Completed 20/256
  Completed 30/256
  Completed 40/256
  Completed 50/256
  Completed 60/256
  Completed 70/256
  Completed 80/256
  Completed 90/256
  Completed 100/256
  Completed 110/256
  Completed 120/256
  Completed 130/256
  Completed 140/256
  Completed 150/256
  Completed 160/256
  Completed 170/256
  Completed 180/256
  Completed 190/256
  Completed 200/256
  Completed 210/256
  Completed 220/256
  Completed 230/256
  Completed 240/256
  Completed 250/256
gemma-3-12b-it benchmark run complete.


In [44]:
results_df = pd.DataFrame(rows)

print("Original results")   
display(results_df.head())
display(results_df.tail())

results_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f'Saved raw CSV: {RAW_RESULTS_PATH}')

Original results


,Time,model_label,model_name,endpoint_port,prompt_version,raw_response,t+1,llm_prediction,llm_confidence,parse_ok,parse_error,ground_truth,ml_prediction
0,2025-02-12 02:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4981.276052,NaN
1,2025-02-12 03:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4980.895880,4981.380412
2,2025-02-12 04:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4981.522123,4980.847361
3,2025-02-12 05:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4981.172066,4981.444753
4,2025-02-12 06:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4980.811139,4981.104882


,Time,model_label,model_name,endpoint_port,prompt_version,raw_response,t+1,llm_prediction,llm_confidence,parse_ok,parse_error,ground_truth,ml_prediction
251,2025-02-22 13:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4950.528957,4951.339960
252,2025-02-22 14:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4952.101775,4950.599438
253,2025-02-22 15:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4952.104437,4952.357578
254,2025-02-22 16:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4951.670969,4952.028750
255,2025-02-22 17:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,"```json\n{\n""t+1"": ""2/12/2025 12:00:00 AM"",\n""...",2/12/2025 12:00:00 AM,4982.4312,0.88,True,None,4951.120372,NaN


Saved raw CSV: /home/jupyter-theresa.pham/ESSD_AI_Competition_1/SubTerraTensorTinkerers/Results/gemma3_results_raw_1.csv


In [45]:
try: 
    clean_df = results_df.copy()
except:
    results_df = pd.read_csv(RAW_RESULTS_PATH)
    clean_df = results_df.copy()
    print(f"Loaded raw results from {CLEAN_RESULTS_PATH}")

clean_df[['llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error']] = clean_df[['llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error']].shift(1)
clean_df = clean_df.iloc[1:-1]
clean_df = clean_df.drop(columns=["raw_response", "t+1", "parse_ok", "parse_error"])

print("Clean results")
display(clean_df.head())

clean_df.to_csv(CLEAN_RESULTS_PATH, index=False)
print(f'Saved clean CSV: {CLEAN_RESULTS_PATH}')

Clean results


,Time,model_label,model_name,endpoint_port,prompt_version,llm_prediction,llm_confidence,ground_truth,ml_prediction
1,2025-02-12 03:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4980.895880,4981.380412
2,2025-02-12 04:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4981.522123,4980.847361
3,2025-02-12 05:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4981.172066,4981.444753
4,2025-02-12 06:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4980.811139,4981.104882
5,2025-02-12 07:00:00,gemma-3-12b-it,gemma-3-12b-it,8002,v1,4982.4312,0.88,4979.714603,4980.695511


Saved clean CSV: /home/jupyter-theresa.pham/ESSD_AI_Competition_1/SubTerraTensorTinkerers/Results/gemma3_results_clean_1.csv


## Step 7: Evaluate and compare

What this does: computes ML error, LLM error, and invalid output rate from the parsed results.

Why it matters: this gives a direct Week 3 vs Week 4 comparison on the same data split.

Principle: evaluate both quality and reliability. Accuracy without format reliability can still fail in production workflows.

In [46]:
try: 
    eval_df = clean_df.copy()
except:
    clean_df = pd.read_csv(CLEAN_RESULTS_PATH)
    eval_df = clean_df.copy()
    print(f"Loaded cleaned results from {CLEAN_RESULTS_PATH}")


def reg_metrics(y_true, y_pred):
    return {
        "rmse": np.sqrt(np.mean((y_true - y_pred)**2)),
        "mae": np.mean(np.abs(y_true - y_pred)),
    }

summary_rows = []
group_cols = ["model_label", "model_name", "endpoint_port"]

for keys, g in eval_df.groupby(group_cols, dropna=False):
    valid_llm = g.dropna(subset=["ground_truth", "llm_prediction"])

    # Compare ML and LLM on the exact same valid rows for fairness
    valid_ml = valid_llm.dropna(subset=["ml_prediction"])
    if len(valid_ml) > 0:
        ml_m = reg_metrics(valid_ml["ground_truth"], valid_ml["ml_prediction"])
    else:
        ml_m = {"rmse": np.nan, "mae": np.nan}

    summary_rows.append({
        "model_type": "Week3_ML",
        "model_label": "week3-xgb",
        "model_name": "Week3 ML - XGB",
        "endpoint_port": np.nan,
        "rows_total": int(len(g)),
        "rows_valid": int(len(valid_ml)),
        "invalid_rate": float(1 - len(valid_ml) / len(g)),
        "rmse": ml_m["rmse"],
        "mae": ml_m["mae"]
    })
    
    llm_m = reg_metrics(valid_llm["ground_truth"], valid_llm["llm_prediction"])
    
    summary_rows.append({
        "model_type": "Week4_LLM",
        "model_label": keys[0],
        "model_name": keys[1],
        "endpoint_port": keys[2],
        "rows_total": int(len(g)),
        "rows_valid": int(len(valid_llm)),
        "invalid_rate": float(1 - len(valid_llm) / len(g)),
        "rmse": llm_m["rmse"],
        "mae": llm_m["mae"],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
                                
summary_df.to_csv(SUMMARY_RESULTS_PATH, index=False)
print(f'Saved summary CSV: {SUMMARY_RESULTS_PATH}')

,model_type,model_label,model_name,endpoint_port,rows_total,rows_valid,invalid_rate,rmse,mae
0,Week3_ML,week3-xgb,Week3 ML - XGB,NaN,254,254,0.0,1.027400,0.740474
1,Week4_LLM,gemma-3-12b-it,gemma-3-12b-it,8002.0,254,254,0.0,18.505921,17.028993


Saved summary CSV: /home/jupyter-theresa.pham/ESSD_AI_Competition_1/SubTerraTensorTinkerers/Results/gemma3_results_summary_1.csv
